In [27]:
import pandas as pd
import numpy as np
import pickle

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

Inlezen data

In [28]:
kijkcijfers = pd.read_csv('./data2/feat_eng/kijkcijfers_weerdata.csv')
# SE = sentence embedding
kijkcijfers_SE = pd.read_csv('./data2/feat_eng/kijkcijfers_weerdata_met_sentence_embedding.csv')

# Opsplitsen in train en test set

In [29]:
cutoff = pd.Timestamp('2023-01-01')

kijkcijfers['timestamp'] = pd.to_datetime(kijkcijfers['timestamp'])

train_set_temp = kijkcijfers[kijkcijfers['timestamp'] < cutoff]
test_set_temp = kijkcijfers[kijkcijfers['timestamp'] >= cutoff]

train_set = train_set_temp.select_dtypes(include=[np.number])
test_set = test_set_temp.select_dtypes(include=[np.number])

X_train = train_set.drop(columns=['viewers'])
y_train = train_set['viewers']

X_test = test_set.drop(columns=['viewers'])
y_test = test_set['viewers']

In [30]:
cutoff = pd.Timestamp('2023-01-01')

kijkcijfers_SE['timestamp'] = pd.to_datetime(kijkcijfers_SE['timestamp'])

train_set_temp_SE = kijkcijfers_SE[kijkcijfers_SE['timestamp'] < cutoff]
test_set_temp_SE = kijkcijfers_SE[kijkcijfers_SE['timestamp'] >= cutoff]

train_set_SE = train_set_temp_SE.select_dtypes(include=[np.number])
test_set_SE = test_set_temp_SE.select_dtypes(include=[np.number])

X_train_SE = train_set_SE.drop(columns=['viewers'])
y_train_SE = train_set_SE['viewers']

X_test_SE = test_set_SE.drop(columns=['viewers'])
y_test_SE = test_set_SE['viewers']

Hulpfuncties om modellen te testen

In [31]:
def test_model(model, X, y, n_splits=5, scoring='neg_mean_absolute_error'):
    # Pipeline die voor trainen standard scaling toepast
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    # Om in plaats van gewone folds, gebruik te maken van TimeSeriesSplit om trends in de tijd te behouden
    tscv = TimeSeriesSplit(n_splits=n_splits)
    
    # Voer cross-validatie uit
    scores = cross_val_score(pipeline, X, y, cv=tscv, scoring=scoring, n_jobs=-1)
    
    # Gemiddelde en standaarddeviatie van de scores
    mean_score = -np.mean(scores)
    std_score = np.std(scores)
    
    print(f"Mean MAE: {mean_score:.3f} (+/- {std_score:.3f})")
    return scores

def test_model_list(list, X, y, n_splits=5, scoring='neg_mean_absolute_error'):
    for model in list:
        print(f"Testing model: {model.__class__.__name__}")
        test_model(model, X, y, n_splits=n_splits, scoring=scoring)

# Baseline models testing

In [32]:
ridge_reg = Ridge()

In [34]:
scores = test_model(ridge_reg, X_train, y_train)
scores_SE = test_model(ridge_reg, X_train_SE, y_train_SE)

print(f'Scores: {scores}\nScores_SE: {scores_SE}')

Mean MAE: 92802.968 (+/- 9754.346)
Mean MAE: 93545.440 (+/- 9659.534)
Scores: [ -91917.14356837  -82601.20665302 -110447.43883076  -93886.40246874
  -85162.64653895]
Scores_SE: [ -96028.6528266   -79922.26604413 -109725.85583179  -90659.9410735
  -91390.48277668]


# Other models testing

In [35]:
rf_reg = RandomForestRegressor()
gb_reg = GradientBoostingRegressor()
xgb_reg = XGBRegressor()

In [36]:
print("Without sentence embedding")
test_model_list([rf_reg, gb_reg, xgb_reg], X_train, y_train)

print("\nWith sentence embedding")
test_model_list([rf_reg, gb_reg, xgb_reg], X_train_SE, y_train_SE)

Without sentence embedding
Testing model: RandomForestRegressor
Mean MAE: 62890.985 (+/- 4959.426)
Testing model: GradientBoostingRegressor
Mean MAE: 69094.292 (+/- 5034.250)
Testing model: XGBRegressor
Mean MAE: 63209.376 (+/- 6686.836)

With sentence embedding
Testing model: RandomForestRegressor
Mean MAE: 62737.096 (+/- 4780.647)
Testing model: GradientBoostingRegressor
Mean MAE: 69110.550 (+/- 5595.626)
Testing model: XGBRegressor
Mean MAE: 65609.922 (+/- 7339.826)


Aan de hand van deze resultaten zal ik verder een RandomForestRegressor en een XGBRegressor model proberen finetunen.\
De enige opmerking die ik heb is dat het trainen van de RandomForestRegressor zeer lang duurde.

In [ ]:
y_train.describe()

count    4.508700e+04
mean     4.555592e+05
std      2.831226e+05
min      1.588700e+04
25%      2.367115e+05
50%      3.682120e+05
75%      6.156235e+05
max      2.494114e+06
Name: viewers, dtype: float64

# Finetuning